In [1]:
!git clone https://github.com/gauriiiiiiiiiiii/PatientTrialMapper.git

Cloning into 'PatientTrialMapper'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 43 (delta 17), reused 35 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 32.87 KiB | 1.83 MiB/s, done.
Resolving deltas: 100% (17/17), done.


In [2]:
import os
os.chdir('/kaggle/working/PatientTrialMapper')
print("Current directory:", os.getcwd())

Current directory: /kaggle/working/PatientTrialMapper


In [3]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 108.0 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires nu

In [4]:
!nvidia-smi

Sun Jun  7 09:08:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!git clone https://github.com/gauriiiiiiiiiiii/PatientTrialMapper.git /kaggle/working/PatientTrialMapper
%cd /kaggle/working/PatientTrialMapper
!ls

fatal: destination path '/kaggle/working/PatientTrialMapper' already exists and is not an empty directory.
/kaggle/working
app.py	      Dockerfile   predict.py	   requirements.txt
config.py     evaluate.py  push_to_hub.py  train.py
data_prep.py  pages	   README.md


In [6]:
!python data_prep.py

Built-in examples: 20

Generating synthetic data (skipped if no API key)...
OPENAI_API_KEY not set in config.py — skipping synthetic generation.

── Label distribution ──
response
Eligible        10
Not Eligible    10

── Prompt length stats ──
  min=303  max=371  mean=330

EDA plot saved → data/eda.png

Split → train:16  val:2  test:2
Saving the dataset (1/1 shards): 100%|████| 2/2 [00:00<00:00, 819.84 examples/s]

Saved:
  data/train
  data/val
  data/test_samples.json

data_prep.py done. Run: python train.py


In [7]:
!pip install trl -q

In [8]:
!pip install -U bitsandbytes>=0.46.1 -q

In [9]:
!git pull
!python train.py

Already up to date.
GPU: Tesla T4  (15.6 GB VRAM)

Loading datasets...
  Train: 16  Val: 2

Loading tokenizer...
config.json: 100%|█████████████████████████████| 601/601 [00:00<00:00, 3.45MB/s]
tokenizer_config.json: 141kB [00:00, 58.2MB/s]
tokenizer.json: 1.96MB [00:00, 46.2MB/s]
tokenizer.model: 100%|████████████████████████| 587k/587k [00:00<00:00, 914kB/s]
special_tokens_map.json: 100%|█████████████████| 414/414 [00:00<00:00, 2.98MB/s]

Loading base model: mistralai/Mistral-7B-Instruct-v0.3
(This downloads ~14 GB on first run — be patient)
model.safetensors.index.json: 23.9kB [00:00, 53.3MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [00:58<00:00, 19.37s/it]
Download complete: 100%|████████████████████| 14.5G/14.5G [00:58<00:00, 249MB/s]
Loading weights: 100%|█| 291/291 [00:17<00:00, 17.00it/s, Materializing param=mo
generation_config.json: 100%|███████████████████| 116/116 [00:00<00:00, 420kB/s]

Applying QLoRA...
trainable params: 41,943,040 || all params: 7,289,96

In [10]:
import os
# Check if adapters were saved
if os.path.exists('/kaggle/working/PatientTrialMapper/models/lora_adapters'):
    files = os.listdir('/kaggle/working/PatientTrialMapper/models/lora_adapters')
    print("Adapters saved:", files)
else:
    print("Adapters NOT found - training may have crashed")
    
# Check checkpoints
if os.path.exists('/kaggle/working/PatientTrialMapper/models/checkpoints'):
    ckpts = os.listdir('/kaggle/working/PatientTrialMapper/models/checkpoints')
    print("Checkpoints:", ckpts)

Adapters saved: ['README.md', 'adapter_model.safetensors', 'tokenizer_config.json', 'chat_template.jinja', 'tokenizer.json', 'adapter_config.json']
Checkpoints: ['README.md', 'checkpoint-300', 'checkpoint-500', 'checkpoint-400', 'checkpoint-200', 'checkpoint-100']


In [11]:
!python evaluate.py

Test set loaded: 2 examples

Loading fine-tuned model...
Loading weights: 100%|█| 291/291 [00:10<00:00, 28.44it/s, Materializing param=mo

Running predictions on 2 test examples...
  [1/2]  True: Not Eligible     Pred: Eligible
  [2/2]  True: Eligible         Pred: Eligible

  EVALUATION RESULTS
  Total test samples : 2

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/skl

In [12]:
import shutil
shutil.make_archive('/kaggle/working/lora_adapters', 'zip', 
                    '/kaggle/working/PatientTrialMapper/models/lora_adapters')
print("Download this: /kaggle/working/lora_adapters.zip")

Download this: /kaggle/working/lora_adapters.zip
